# Conditional Probability, Bayes Rule, And Sampling

**Official MA1001B Alignment:** *1.3 conditional probability and Bayes' rule; 1.4 random sampling.*


## How To Use This Lesson

This notebook is designed as a guided teaching episode and interactive lab, not a passive code demonstration. To get the most out of this lesson:
1. **Read the conceptual explanations and explicit links** before running any code.
2. **Execute code cells sequentially**, paying attention to inline educational comments.
3. **Pause at the Guided Checkpoint** to discuss with a partner and write your reasoning before checking solutions.
4. **Complete the Independent Practice and Exit Ticket**; written justification is the primary evidence of statistical competence.


## Learning Goals

By the end of this lesson, you will be able to:
- Distinguish between conditional probability `P(A|B)` and its reverse `P(B|A)` in practical decision contexts.
- Calculate marginal, joint, and conditional probabilities from tabular data using Pandas filtering and grouping.
- Apply Bayes' rule to invert conditional probabilities and incorporate population base rates.
- Evaluate sample-to-sample variability in conditional probability estimates across random sub-samples.


## The Three Explicit Links

In accordance with the MA1001B pedagogical framework, this lesson explicitly connects theory, computation, and action:

- **1. Conceptual Link (What is modeled):** We model risk conditional on demographic and environmental factors to understand relative vulnerability and base rates.
- **2. Computational Link (How Python represents it):** We use Pandas Boolean indexing and cross-tabulations (`pd.crosstab`, `.groupby`) to isolate conditional reference groups.
- **3. Decision Link (How it guides action):** Precise conditional risk communication prevents policy misallocation caused by confounding base rates with conditional risk.


## Decision Scenario

> **The Problem:** A safety analyst is asked to communicate risk from passenger data. The analyst must distinguish P(survived | group) from P(group | survived), because they answer different questions.


## Conceptual Explanation

Conditional probability restricts the reference group. P(A | B) means the probability of A among cases where B is true. Bayes' rule lets us reverse a conditional probability when we also know the base rates. In data science, many communication errors come from switching the condition and the outcome.


## Mathematical Anchor

P(A | B) = P(A intersection B) / P(B). Bayes' rule: P(A | B) = P(B | A) P(A) / P(B).


## Data And Workflow Notes

Uses Titanic if `data/raw/titanic/train.csv` exists; otherwise uses a simulated table with the same kind of variables.


## Practical Python Workflow

The following worked example demonstrates how to implement these statistical concepts in Python to generate evidence for decision making.


### Step 1: Data Acquisition & Fallback Simulation

We load the Titanic passenger dataset from `data/raw/` if available; otherwise, we generate a statistically equivalent simulated dataset so the lesson remains standalone runnable.


In [ ]:
# Import required data science and statistical libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set reproducible random seed and visual styling
rng = np.random.default_rng(1001)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

# Load real Titanic data if available, otherwise generate simulated passenger records
path = Path("data/raw/titanic/train.csv")
if path.exists():
    passengers = pd.read_csv(path).rename(
        columns={"Survived": "survived", "Sex": "sex", "Pclass": "pclass"}
    )
else:
    print(f"Notice: Missing {path}. Using fallback simulation. Download Titanic from Kaggle for graded work.")
    passengers = pd.DataFrame({
        "survived": rng.binomial(1, 0.38, 891),
        "sex": rng.choice(["female", "male"], 891, p=[0.36, 0.64]),
        "pclass": rng.choice([1, 2, 3], 891, p=[0.24, 0.21, 0.55]),
    })

passengers[["survived", "sex", "pclass"]].head()


### Step 2: Marginal vs. Conditional Probability

We calculate and contrast marginal survival rate P(survived), conditional survival rate given sex P(survived | female), and the reverse conditional probability P(female | survived).


In [ ]:
# Define Boolean masks for survival and sex
survived = passengers["survived"].eq(1)
female = passengers["sex"].eq("female")

# Contrast marginal, conditional, and reverse conditional probabilities
pd.Series({
    "P(survived) [Marginal]": survived.mean(),
    "P(female) [Base Rate]": female.mean(),
    "P(survived | female) [Risk given group]": survived[female].mean(),
    "P(female | survived) [Group composition of survivors]": female[survived].mean(),
}).round(3)


### Step 3: Multi-Variable Conditional Risk Table

We group passengers by both sex and passenger class to analyze how conditional survival rates vary across intersecting demographic and socio-economic segments.


In [ ]:
# Compute conditional survival rates across sex and passenger class
conditional_table = (
    passengers
    .groupby(["sex", "pclass"])["survived"]
    .agg(total_passengers="count", survivors="sum", survival_rate="mean")
    .sort_values("survival_rate", ascending=False)
)
conditional_table.round(3)


### Step 4: Sampling Variability in Conditional Estimates

We draw 30 random sub-samples (n=120 each) from the dataset to observe how much conditional probability estimates fluctuate due to sampling error.


In [ ]:
# Simulate drawing 30 random samples of 120 passengers to check estimation stability
sample_estimates = []
for seed in range(30):
    sampled = passengers.sample(120, random_state=seed)
    sample_estimates.append(sampled.loc[sampled["sex"].eq("female"), "survived"].mean())

# Summarize the distribution of conditional survival estimates across samples
pd.Series(sample_estimates, name="estimated_P(survived|female)").describe().round(3)


## Guided Checkpoint

> [!IMPORTANT]
> **Pair Discussion & Writing Prompt:**
> Write two sentences: one using P(survived | female) correctly and one using P(female | survived) correctly.

*Write your reasoned response below before continuing:*


## Common Mistakes & Statistical Pitfalls

Avoid these frequent errors when conducting or communicating this analysis:
- **Warning:** Reversing the condition and the outcome (the Prosecutor's Fallacy).
- **Warning:** Ignoring population base rates when interpreting conditional risk or Bayes' rule.
- **Warning:** Treating historical statistical associations as definitive proof of a causal survival mechanism.


## Independent Practice

> [!TIP]
> **Your Task:**
> Choose another subgroup (e.g., `pclass == 1`), estimate its conditional survival probability, and compare the full-data estimate with repeated random samples of size 120.

*Use the empty code and markdown cells below to implement your analysis and justify your recommendation.*


In [ ]:
# Write your independent practice code here
# Remember to inspect your outputs and check assumptions


## Decision Interpretation Template

Use this structured format to write your defensible conclusion and recommendation:

1. **The Decision Question:** *State the practical question being answered...*
2. **The Statistical Evidence:** *Summarize key metrics, intervals, p-values, or model comparisons...*
3. **Uncertainty & Limitations:** *Identify what the data cannot prove and what assumptions were made...*
4. **Actionable Recommendation:** *Therefore, I recommend [action] because [justification]...*


## Exit Ticket

> **Reflection:** What critical risk information is lost when a probability is reported without explicitly naming its conditioning group?

*Write your brief conceptual reflection below:*
